# RAG 실험 결과 분석

이미 실행된 RAG 실험 결과를 불러와 비교하고 시각화합니다.
직접 실험을 돌리지 않고 experiments/rag_*/ 폴더에 저장된 산출물만 읽습니다.

사용법: 아래 셀들을 순서대로 실행하세요. HTML로 내보내면 실험 아티팩트로 남길 수 있습니다.

## 1. 실험 목록 불러오기

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import yaml

def find_project_root(start: Path) -> Path:
    markers = ("AGENTS.md", "configs", "scripts", "src")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

PROJECT_ROOT = find_project_root(Path.cwd()).resolve()
EXPERIMENTS_ROOT = PROJECT_ROOT / "experiments"

EXPERIMENT_DIRS = sorted([
    d for d in EXPERIMENTS_ROOT.iterdir()
    if d.is_dir() and (d / "metrics.json").exists()
])

if not EXPERIMENT_DIRS:
    print("metrics.json이 있는 실험 폴더가 없습니다.")
    print("  먼저 run_rag_chat.py --evaluate 를 실행하세요.")
else:
    print(f"발견된 실험: {len(EXPERIMENT_DIRS)}개")
    for d in EXPERIMENT_DIRS:
        print(f"  {d.name}/")

## 2. 실험 선택

In [ ]:
# 전체: 그대로 두면 됨
# 특정: SELECTED = ["rag_langchain", "rag_hybrid"]

SELECTED = [d.name for d in EXPERIMENT_DIRS]
print(f"선택: {SELECTED}")

all_metrics = []
for exp_name in SELECTED:
    mp = EXPERIMENTS_ROOT / exp_name / "metrics.json"
    if mp.exists():
        m = json.loads(mp.read_text(encoding="utf-8"))
        m["experiment"] = exp_name
        all_metrics.append(m)
df_metrics = pd.DataFrame(all_metrics).set_index("experiment")
df_metrics

## 3. 지표 비교 (Bar Chart)

In [ ]:
if not df_metrics.empty:
    metric_names = ["retrieval_hit_rate", "answer_contains_expected_rate",
                    "citation_correct_rate", "not_found_rate"]
    n_exp = len(df_metrics)
    fig, ax = plt.subplots(figsize=(max(8, n_exp * 1.8), 5))
    x = range(n_exp)
    bar_width = 0.2
    colors = ["#4CAF50", "#2196F3", "#FF9800", "#F44336"]
    for i, (metric, color) in enumerate(zip(metric_names, colors)):
        values = [df_metrics.loc[name, metric] if name in df_metrics.index else 0
                  for name in SELECTED]
        bars = ax.bar([pos + i * bar_width for pos in x], values, bar_width,
                      label=metric, color=color)
        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.02, f"{val:.2f}",
                        ha="center", fontsize=7)
    ax.set_ylabel("score")
    ax.set_title("RAG Metrics Comparison")
    ax.set_xticks([pos + bar_width * 1.5 for pos in x])
    ax.set_xticklabels(SELECTED, rotation=20, ha="right")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_ylim(0, 1.15)
    plt.tight_layout()
    plt.show()

## 4. 2개 실험 집중 비교 (Delta)

In [ ]:
if len(SELECTED) >= 2:
    A, B = SELECTED[0], SELECTED[1]
else:
    A, B = SELECTED[0], SELECTED[0]
print(f"Before: {A}  /  After: {B}")

if A in df_metrics.index and B in df_metrics.index:
    rows = []
    for metric in ["retrieval_hit_rate", "answer_contains_expected_rate",
                   "citation_correct_rate", "not_found_rate"]:
        a_val, b_val = df_metrics.loc[A, metric], df_metrics.loc[B, metric]
        delta = b_val - a_val
        emoji = "+ " if delta > 0.03 else "- " if delta < -0.03 else "= "
        rows.append({
            "metric": metric, A: f"{a_val:.3f}", B: f"{b_val:.3f}",
            "delta": f"{delta:+.3f} {emoji}",
        })
    display(pd.DataFrame(rows))

## 5. Config 차이점 비교

실험 간 Config가 어떻게 다른지 보여줍니다.

In [ ]:
def load_config_yaml(exp_name):
    path = EXPERIMENTS_ROOT / exp_name / "config.yaml"
    if path.exists():
        return yaml.safe_load(path.read_text(encoding="utf-8"))
    return {}

if len(SELECTED) >= 2:
    configs = {name: load_config_yaml(name) for name in SELECTED[:4]}
    # RAG 관련 주요 키만 비교
    keys_of_interest = [
        ("rag", "engine"),
        ("rag", "splitter", "chunk_size"),
        ("rag", "splitter", "chunk_overlap"),
        ("rag", "embedding", "provider"),
        ("rag", "retriever", "method"),
        ("rag", "retriever", "top_k"),
        ("rag", "answerer", "provider"),
        ("rag", "reranker", "enabled"),
    ]
    config_rows = []
    for key_path in keys_of_interest:
        label = ".".join(key_path)
        row = {"config_key": label}
        for name in SELECTED[:4]:
            val = configs.get(name, {})
            for k in key_path:
                val = val.get(k, {}) if isinstance(val, dict) else val
            row[name] = str(val) if val is not None else "-"
        config_rows.append(row)
    df_config = pd.DataFrame(config_rows)
    display(df_config)
else:
    print("비교할 실험이 2개 이상 필요합니다.")

## 6. 실험 간 답변 텍스트 비교

같은 질문에 대해 각 실험이 어떤 답변을 했는지 side-by-side로 봅니다.

In [ ]:
all_answers = {}
for exp_name in SELECTED[:4]:
    ans_path = EXPERIMENTS_ROOT / exp_name / "answers.jsonl"
    if ans_path.exists():
        lines = ans_path.read_text(encoding="utf-8").splitlines()
        answers = [json.loads(line) for line in lines if line.strip()]
        all_answers[exp_name] = {a["question"]: a for a in answers}

if len(all_answers) >= 2:
    exp_names = list(all_answers.keys())
    # 모든 실험에 공통으로 있는 질문만
    common_questions = set.intersection(*[set(a.keys()) for a in all_answers.values()])
    common_questions = sorted(common_questions)[:10]  # 최대 10개

    if common_questions:
        for q in common_questions:
            print(f"질문: {q}")
            print("-" * 60)
            for exp_name in exp_names:
                a = all_answers[exp_name].get(q, {})
                ans_text = a.get("answer", "?")[:100]
                status = a.get("status", "?")
                print(f"  [{exp_name}] ({status}): {ans_text}")
            print()
    else:
        print("공통 질문이 없습니다 (실험마다 평가셋이 다를 수 있음)")
else:
    print("비교할 실험이 2개 이상 필요합니다.")

## 7. 질문별 hit/miss 히트맵

어떤 질문 유형이 어떤 실험에서 실패했는지 한눈에.

In [ ]:
all_evals = {}
for exp_name in SELECTED[:6]:
    eval_path = EXPERIMENTS_ROOT / exp_name / "evaluation_results.csv"
    if eval_path.exists():
        df = pd.read_csv(eval_path)
        all_evals[exp_name] = df

if len(all_evals) >= 2:
    # 모든 실험에 공통인 질문만 모으기
    common_qs = set.intersection(*[set(df["question"]) for df in all_evals.values()])
    common_qs = sorted(common_qs)[:15]
    if common_qs:
        heatmap_data = {}
        for exp_name, df in all_evals.items():
            df_q = df[df["question"].isin(common_qs)].set_index("question")
            hit_values = (df_q["retrieval_hit"] == "true").astype(int)
            heatmap_data[exp_name] = hit_values
        df_heat = pd.DataFrame(heatmap_data)

        fig, ax = plt.subplots(figsize=(max(6, len(df_heat.columns)*1.5), max(4, len(df_heat)*0.35)))
        im = ax.imshow(df_heat.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
        ax.set_xticks(range(len(df_heat.columns)))
        ax.set_xticklabels(df_heat.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(df_heat.index)))
        ax.set_yticklabels([q[:40] for q in df_heat.index], fontsize=9)
        ax.set_title("Retrieval Hit Heatmap")
        for i in range(len(df_heat.index)):
            for j in range(len(df_heat.columns)):
                val = df_heat.values[i, j]
                ax.text(j, i, "HIT" if val else "MISS", ha="center", va="center",
                        fontsize=7, color="black" if val else "darkred")
        plt.tight_layout()
        plt.show()
    else:
        print("공통 질문이 없습니다.")
else:
    print("evaluation_results.csv가 있는 실험이 2개 이상 필요합니다.")

## 8. Retriever 방식별 비교

In [ ]:
COMPARISON_CSV = PROJECT_ROOT / "reports" / "rag_retriever_comparison.csv"
if COMPARISON_CSV.exists():
    df_comp = pd.read_csv(COMPARISON_CSV)
    display(df_comp)
    if not df_comp.empty and "experiment" in df_comp.columns:
        metric_cols = [c for c in ["retrieval_hit_rate",
                       "answer_contains_expected_rate", "citation_correct_rate"]
                       if c in df_comp.columns]
        if metric_cols:
            df_comp.set_index("experiment")[metric_cols].plot.bar(
                figsize=(10, 5), color=["#4CAF50", "#2196F3", "#FF9800"],
                title="Retriever Comparison")
            plt.ylabel("score"); plt.ylim(0, 1.1)
            plt.xticks(rotation=0); plt.tight_layout(); plt.show()
else:
    print("rag_retriever_comparison.csv 없음")
    print("  python scripts/compare_rag_retrievers.py --project-root .")

## 9. 실패 분석 통합

In [ ]:
failure_summary = []
failure_types = ["bad_retrievals", "unsupported_answers", "failed_questions"]
for exp_name in SELECTED:
    row = {"experiment": exp_name}
    for ftype in failure_types:
        path = EXPERIMENTS_ROOT / exp_name / f"{ftype}.csv"
        row[ftype] = len(pd.read_csv(path)) if path.exists() else 0
    failure_summary.append(row)

df_fail = pd.DataFrame(failure_summary).set_index("experiment")
if df_fail.empty or df_fail.sum().sum() == 0:
    print("모든 실험에서 실패 케이스가 없습니다!")
else:
    display(df_fail)
    df_fail.plot.bar(figsize=(10, 4), color=["#FF9800", "#2196F3", "#F44336"],
                     title="Failure Case Breakdown")
    plt.ylabel("count"); plt.xticks(rotation=20, ha="right")
    plt.tight_layout(); plt.show()

## 10. 특정 실험 상세 진단

In [ ]:
DETAIL_EXP = SELECTED[0]
DETAIL_DIR = EXPERIMENTS_ROOT / DETAIL_EXP
print(f"분석 대상: {DETAIL_EXP}")

# 검색 결과 미리보기
ret_path = DETAIL_DIR / "retrieval_results.jsonl"
if ret_path.exists():
    rows = [json.loads(line) for line in
            ret_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"\n=== 검색 결과 (총 {len(rows)}건, 첫 3건) ===")
    for r in rows[:3]:
        print(f"  Q: {r.get('question', '?')}")
        for chunk in r.get("retrieved_chunks", [])[:2]:
            print(f"    [{chunk.get('rank','?')}등] score={chunk.get('score',0):.3f}"
                  f"  {chunk.get('chunk_id','?')}: {chunk.get('text','')[:60]}...")
        print()

# 검색 실패 질문
bad_path = DETAIL_DIR / "bad_retrievals.csv"
if bad_path.exists():
    bad = pd.read_csv(bad_path)
    if not bad.empty:
        print(f"=== 검색 실패 질문 ({len(bad)}건) ===")
        for _, row in bad.head(5).iterrows():
            print(f"  X {row.get('question','?')}")
            print(f"    expected_answer: {str(row.get('expected_answer','?'))[:60]}")

# 답변 상태 분포
ans_path = DETAIL_DIR / "answers.jsonl"
if ans_path.exists():
    answers = [json.loads(line) for line in
               ans_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    from collections import Counter
    print(f"\n=== 답변 상태 분포 ===")
    for status, count in Counter(a.get("status","?") for a in answers).items():
        print(f"  {status}: {count}건")

## 11. 저장

File > Download as > HTML 로 내보내면 실험 리포트가 됩니다.

In [ ]:
print("=== 실험 요약 ===")
print(f"비교 실험: {len(SELECTED)}개")
if not df_metrics.empty:
    for metric in ["retrieval_hit_rate", "answer_contains_expected_rate"]:
        if metric in df_metrics.columns:
            best_idx = df_metrics[metric].idxmax()
            best_val = df_metrics.loc[best_idx, metric]
            print(f"  최고 {metric}: {best_idx} = {best_val:.3f}")
print(f"분석 일시: {pd.Timestamp.now()}")